In [3]:
print("working")
!pip install yaml

working


ERROR: Could not find a version that satisfies the requirement yaml (from versions: none)
ERROR: No matching distribution found for yaml


In [7]:
import os
import torch
import pandas as pd
import numpy as np
from data_loader import convert_all_raw_data
from preprocessing import run_preprocessing_pipeline, load_config
from features import run_feature_engineering
# Import new DL modules
from deep_learning import RUL_LSTM, prepare_lstm_data, train_model_dl, evaluate_lstm

def main():
    # 1. Path Configuration
    raw_dir, interim_dir = "../data/raw", "../data/interim"
    os.makedirs("../data/processed", exist_ok=True)
    
    # 2. Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = load_config()
    datasets = ['FD001', 'FD002', 'FD003', 'FD004']
    all_results = []
    
    # Deep Learning Hyperparameters 
    WINDOW_SIZE = 30  # Look back 30 cycles
    BATCH_SIZE = 64
    EPOCHS = 50
    
    print(f"🚀 Step 1: Data Setup (Device: {device})")
    convert_all_raw_data(raw_dir, interim_dir)
    
    print("\n🚀 Step 2: Sequential Deep Learning Pipeline")
    
    for ds in datasets:
        print(f"\n📦 Processing Dataset: {ds}")
        
        # Load Raw Interim Data
        train_df = pd.read_csv(f"{interim_dir}/train_{ds}.csv")
        test_df = pd.read_csv(f"{interim_dir}/test_{ds}.csv")
        y_truth = pd.read_csv(f"{interim_dir}/RUL_{ds}.csv")['RUL'].values
        
        # A. Preprocessing & B. Feature Engineering (Reuse existing logic)
        train_proc, fitted_models = run_preprocessing_pipeline(train_df, ds, config)
        test_proc, _ = run_preprocessing_pipeline(test_df, ds, config, fitted_models=fitted_models)
        
        train_final = run_feature_engineering(train_proc, ds)
        test_final = run_feature_engineering(test_proc, ds)
        
        # Identify sensors for input
        drop_cols = ['unit_id', 'time', 'regime_id', 'RUL', 'RUL_clipped']
        features = [c for c in train_final.columns if c not in drop_cols]
        
        # C. Sequence Generation (Reshaping for LSTM)
        print(f"   Generating {WINDOW_SIZE}-cycle sequences...")
        X_train, y_train = prepare_lstm_data(train_final, WINDOW_SIZE, features, 'RUL_clipped')
        
        # D. Training the LSTM
        print(f"   Training LSTM (Epochs: {EPOCHS})...")
        model = RUL_LSTM(input_dim=len(features)).to(device)
        model = train_model_dl(X_train, y_train, model, batch_size=BATCH_SIZE, epochs=EPOCHS)
        
        # E. Evaluation (Predicting on the terminal sequence)
        rmse, score = evaluate_lstm(model, test_final, y_truth, features, WINDOW_SIZE)
        
        all_results.append({'Dataset': ds, 'RMSE': round(rmse, 2), 'NASA Score': round(score, 2)})
        print(f"   ✅ {ds} Metrics -> RMSE: {rmse:.2f} | Score: {score:.2f}")

    # Final Summary
    summary_df = pd.DataFrame(all_results)
    print("\n" + "."*10 + "\n" + "       LSTM PERFORMANCE SUMMARY" + "\n" + "."*10)
    print(summary_df.to_string(index=False))
    print("."*10)

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'yaml'

In [ ]:
import os
import yaml
import pandas as pd
from src.data_loader import convert_all_raw_data
from src.preprocessing import run_preprocessing_pipeline, load_config
from src.features import run_feature_engineering
from src.modeling import train_model, evaluate_on_test

def main():
    # 1. Path Configuration
    raw_dir = "data/raw"
    interim_dir = "data/interim"
    processed_dir = "data/processed"
    
    # Ensure the processed directory exists before we try to save to it
    os.makedirs(processed_dir, exist_ok=True)
    
    print("🚀 Step 1: Converting Raw TXT to Interim CSV...")
    convert_all_raw_data(raw_dir, interim_dir)
    
    config = load_config()
    datasets = ['FD001', 'FD002', 'FD003', 'FD004']
    all_results = []

    print("\n🚀 Step 2: Running End-to-End Pipeline...")
    
    for ds in datasets:
        print(f"\n📦 Processing Dataset: {ds}")
        
        # Load from Interim (Created in Step 1)
        train_df = pd.read_csv(f"{interim_dir}/train_{ds}.csv")
        test_df = pd.read_csv(f"{interim_dir}/test_{ds}.csv")
        y_truth = pd.read_csv(f"{interim_dir}/RUL_{ds}.csv")['RUL'].values
        
        # # A. Preprocessing (In-Memory)
        # train_proc = run_preprocessing_pipeline(train_df, ds, config)
        # test_proc = run_preprocessing_pipeline(test_df, ds, config)
        
        # # B. Feature Engineering (In-Memory)
        # train_final = run_feature_engineering(train_proc, ds)
        # test_final = run_feature_engineering(test_proc, ds)
        
        # # C. Save Final Features (Populates your processed folder)
        # train_final.to_csv(f"{processed_dir}/train_{ds}_final.csv", index=False)
        # test_final.to_csv(f"{processed_dir}/test_{ds}_final.csv", index=False)
        # print(f"   💾 Saved final features to {processed_dir}")

        # A. Process Training Data (Fits the models)
        train_proc, fitted_models = run_preprocessing_pipeline(train_df, ds, config, fitted_models=None)

        # B. Process Test Data (Uses training models)
        test_proc, _ = run_preprocessing_pipeline(test_df, ds, config, fitted_models=fitted_models)

        # C. Feature Engineering (Proceed as normal)
        train_final = run_feature_engineering(train_proc, ds)
        test_final = run_feature_engineering(test_proc, ds)
                
        # D. Modeling
        drop_cols = ['unit_id', 'time', 'regime_id', 'RUL', 'RUL_clipped']
        features = [c for c in train_final.columns if c not in drop_cols]
        
        model = train_model(train_final[features], train_final['RUL_clipped'], ds)
        
        # E. Evaluation
        rmse, score, y_pred = evaluate_on_test(model, test_final, y_truth, features)
        
        all_results.append({
            'Dataset': ds, 'RMSE': round(rmse, 2), 'NASA Score': round(score, 2)
        })
        print(f"   ✅ {ds} Complete | RMSE: {rmse:.2f} | Score: {score:.2f}")

    # Final Summary
    summary_df = pd.DataFrame(all_results)
    print("\n" + "="*45)
    print("         TURBOFAN PROJECT SUMMARY")
    print("="*45)
    print(summary_df.to_string(index=False))
    print("="*45)

if __name__ == "__main__":
    main()

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from src.data_loader import convert_all_raw_data
from src.preprocessing import run_preprocessing_pipeline, load_config
from src.features import run_feature_engineering
# Import new DL modules
from src.deep_learning import RUL_LSTM, prepare_lstm_data, train_model_dl, evaluate_lstm

def main():
    # 1. Path Configuration
    raw_dir, interim_dir = "data/raw", "data/interim"
    os.makedirs("data/processed", exist_ok=True)
    
    # 2. Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = load_config()
    datasets = ['FD001', 'FD002', 'FD003', 'FD004']
    all_results = []
    
    # Deep Learning Hyperparameters 
    WINDOW_SIZE = 30  # Look back 30 cycles
    BATCH_SIZE = 64
    EPOCHS = 50
    
    print(f"🚀 Step 1: Data Setup (Device: {device})")
    convert_all_raw_data(raw_dir, interim_dir)
    
    print("\n🚀 Step 2: Sequential Deep Learning Pipeline")
    
    for ds in datasets:
        print(f"\n📦 Processing Dataset: {ds}")
        
        # Load Raw Interim Data
        train_df = pd.read_csv(f"{interim_dir}/train_{ds}.csv")
        test_df = pd.read_csv(f"{interim_dir}/test_{ds}.csv")
        y_truth = pd.read_csv(f"{interim_dir}/RUL_{ds}.csv")['RUL'].values
        
        # A. Preprocessing & B. Feature Engineering (Reuse existing logic)
        train_proc, fitted_models = run_preprocessing_pipeline(train_df, ds, config)
        test_proc, _ = run_preprocessing_pipeline(test_df, ds, config, fitted_models=fitted_models)
        
        train_final = run_feature_engineering(train_proc, ds)
        test_final = run_feature_engineering(test_proc, ds)
        
        # Identify sensors for input
        drop_cols = ['unit_id', 'time', 'regime_id', 'RUL', 'RUL_clipped']
        features = [c for c in train_final.columns if c not in drop_cols]
        
        # C. Sequence Generation (Reshaping for LSTM)
        print(f"   Generating {WINDOW_SIZE}-cycle sequences...")
        X_train, y_train = prepare_lstm_data(train_final, WINDOW_SIZE, features, 'RUL_clipped')
        
        # D. Training the LSTM
        print(f"   Training LSTM (Epochs: {EPOCHS})...")
        model = RUL_LSTM(input_dim=len(features)).to(device)
        model = train_model_dl(X_train, y_train, model, batch_size=BATCH_SIZE, epochs=EPOCHS)
        
        # E. Evaluation (Predicting on the terminal sequence)
        rmse, score = evaluate_lstm(model, test_final, y_truth, features, WINDOW_SIZE)
        
        all_results.append({'Dataset': ds, 'RMSE': round(rmse, 2), 'NASA Score': round(score, 2)})
        print(f"   ✅ {ds} Metrics -> RMSE: {rmse:.2f} | Score: {score:.2f}")

    # Final Summary
    summary_df = pd.DataFrame(all_results)
    print("\n" + "."*10 + "\n" + "       LSTM PERFORMANCE SUMMARY" + "\n" + "."*10)
    print(summary_df.to_string(index=False))
    print("."*10)

if __name__ == "__main__":
    main()